In [ ]:
# 02_feature_engineering.py (improved)
import os
import re
import json
import time
import tempfile
from pathlib import Path
import pandas as pd
import numpy as np
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import joblib
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()


# --- Config ---
INPUT_PATH = "data/labeled_roles.csv"
OUTPUT_PATH = "data/features.csv"
REPORT_DIR = Path("reports")
FEATURE_STATS_CSV = REPORT_DIR / "feature_stats.csv"
MODEL_DIR = Path("models")
LLM_CACHE_PATH = Path("cache/llm_manager_prob_cache.json")

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LLM_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

# LLM settings (do NOT hardcode keys in source)
USE_LLM_FEATURE = True
GROQ_API_KEY = os.getenv("GROQ_API_KEY")  # set in env, never in source

if USE_LLM_FEATURE:
    try:
        from groq import Groq  # optional SDK
        if not GROQ_API_KEY:
            raise ValueError("GROQ_API_KEY not set in environment")
        groq_client = Groq(api_key=GROQ_API_KEY)
        print("✅ Groq client initialized")
    except ImportError:
        print("⚠️ 'groq' sdk not installed; LLM feature disabled. pip install groq")
        USE_LLM_FEATURE = False
    except Exception as e:
        print(f"⚠️ Groq disabled: {e}")
        USE_LLM_FEATURE = False

# --- Load data ---
input_path = Path(INPUT_PATH)
if not input_path.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_PATH}")
raw_df = pd.read_csv(input_path)
print(f"✅ Loaded {len(raw_df)} samples for feature extraction")

# --- Normalize text ---
if "full_text" not in raw_df.columns:
    raise KeyError("Expected 'full_text' column in input CSV")

raw_df["full_text"] = raw_df["full_text"].fillna("").astype(str)
raw_df["full_text_clean"] = raw_df["full_text"].str.replace(r"\s+", " ", regex=True).str.strip()
text_col = "full_text_clean"

# --- Basic text features ---
df = raw_df.copy()
df["word_count"] = df[text_col].str.split().str.len().fillna(0).astype(int)
df["sentence_count"] = df[text_col].str.count(r"[.!?]+").fillna(0).astype(int)
df["avg_sentence_len"] = np.where(df["sentence_count"] > 0,
                                  df["word_count"] / df["sentence_count"],
                                  0.0)
df["question_count"] = df[text_col].str.count(r"\?").fillna(0).astype(int)

# --- Directive patterns: split hard vs soft ---
HARD_DIRECTIVE_PATTERNS = [
    r"\byou should\b", r"\byou need to\b", r"\bneed to\b",
    r"\bhave to\b", r"\bmake sure\b", r"\bdeadline\b", r"\bblocker\b",
    r"\bassign\b", r"\breview\b.*\bPR\b", r"\bensure\b", r"\bby EOD\b", r"\bI expect\b"
]
SOFT_HELP_PATTERNS = [
    r"\bi['’]?ll help\b", r"\bi will help\b", r"\bi['’]?ll pair\b",
    r"\bpair with you\b", r"\bdm me\b", r"\bmessage me\b", r"\bi can help\b",
    r"\bi can pair\b", r"\bwe can go through\b", r"\bi can walk you through\b"
]

hard_re = re.compile("|".join(HARD_DIRECTIVE_PATTERNS), re.IGNORECASE)
soft_re = re.compile("|".join(SOFT_HELP_PATTERNS), re.IGNORECASE)
df["hard_directive_count"] = df[text_col].apply(lambda x: len(hard_re.findall(x)))
df["soft_help_count"] = df[text_col].apply(lambda x: len(soft_re.findall(x)))
df["directive_count"] = df["hard_directive_count"] + df["soft_help_count"]

# --- Uncertainty / junior-like phrases ---
UNCERTAINTY_PATTERNS = [
    r"\bnot sure\b", r"\bstuck\b", r"\bconfused\b", r"\btrying to\b",
    r"\bi think\b", r"\bmaybe\b", r"\bkind of\b", r"\bsort of\b",
    r"\bdon't know\b", r"\bcan't figure\b", r"\bstruggling\b"
]
uncertainty_re = re.compile("|".join(UNCERTAINTY_PATTERNS), re.IGNORECASE)
df["uncertainty_count"] = df[text_col].apply(lambda x: len(uncertainty_re.findall(x)))

# --- Greeting-only flag (short greetings) ---
greeting_re = re.compile(r"^(hi|hey|hello|morning|good morning|good afternoon|good evening)[\s!.,]*$", re.IGNORECASE)
df["is_greeting_only"] = df[text_col].apply(lambda x: 1 if greeting_re.match(x.strip()) else 0)

# --- Sentiment (cacheable if dataset large) ---
# If dataset grows, replace TextBlob with a faster sentiment lib or cache results.
df["sentiment_score"] = df[text_col].apply(lambda x: TextBlob(x).sentiment.polarity if x else 0.0)

# --- Per-meeting relative features (important) ---
if "meeting_id" not in df.columns:
    raise KeyError("Expected 'meeting_id' in input CSV")
grouped = df.groupby("meeting_id")
df["word_count_rel"] = df["word_count"] / (grouped["word_count"].transform("mean") + 1e-9)
df["directive_count_rel"] = df["directive_count"] / (grouped["directive_count"].transform("mean") + 1e-9)
df["word_count_share"] = df["word_count"] / (grouped["word_count"].transform("sum") + 1e-9)

# ================================
# LLM-derived feature with caching & sanitization
# ================================
def _sanitize_for_prompt(s: str, max_len: int = 800):
    s = s.replace("\n", " ").replace('"', "'")
    return (s[:max_len]).strip()

def _save_cache_atomic(path: Path, cache: dict):
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(cache, f)
    tmp.replace(path)

def get_llm_manager_score_cached(text: str, cache: dict) -> float:
    if text in cache:
        return cache[text]
    if not USE_LLM_FEATURE:
        return 0.5
    sanitized = _sanitize_for_prompt(text)
    try:
        # Respect the API rate limits: consider batching or chunking if dataset large
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{
                "role": "user",
                "content": (
                    "On a scale from 0.0 to 1.0 (only return the number), "
                    f"how managerial is this speaker? Text: \"{sanitized}\""
                )
            }],
            max_tokens=6,
            temperature=0.0,
            top_p=0.5
        )
        raw_output = response.choices[0].message.content.strip()
        match = re.search(r"(\d+(?:\.\d+)?)", raw_output)
        score = float(match.group(1)) if match else 0.5
        score = float(max(0.0, min(1.0, score)))
        cache[text] = score
        _save_cache_atomic(LLM_CACHE_PATH, cache)
        time.sleep(0.08)  # small polite pause
        return score
    except Exception as e:
        print(f"⚠️ Groq API error: {e}")
        return 0.5

# Load cache
if LLM_CACHE_PATH.exists():
    try:
        with open(LLM_CACHE_PATH, "r") as f:
            llm_cache = json.load(f)
    except Exception:
        llm_cache = {}
else:
    llm_cache = {}

# Compute LLM feature in chunks to avoid long running single call (and to log progress)
print(f"🧠 Adding LLM feature (cached entries: {len(llm_cache)})...")
if USE_LLM_FEATURE:
    # iterate rows and fetch only missing entries
    missing_mask = ~df[text_col].isin(llm_cache.keys())
    if missing_mask.any():
        for idx in df[missing_mask].index:
            txt = df.at[idx, text_col]
            llm_cache_val = get_llm_manager_score_cached(txt, llm_cache)
            # value already saved inside function
    # populate column from cache
    df["lm_manager_prob"] = df[text_col].map(lambda t: llm_cache.get(t, 0.5)).astype(float)
else:
    df["lm_manager_prob"] = 0.5

# ================================
# TF-IDF + TruncatedSVD (save artifacts)
# ================================
print("🧠 Extracting TF-IDF features (dim=32)...")
tfidf = TfidfVectorizer(
    max_features=1000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)
tfidf_matrix = tfidf.fit_transform(df[text_col])

# choose SVD dim safely
n_components = 32
max_allowed = min(tfidf_matrix.shape[0] - 1, tfidf_matrix.shape[1] - 1)
n_components = min(n_components, max_allowed) if max_allowed > 0 else 1
svd = TruncatedSVD(n_components=n_components, random_state=42)
tfidf_reduced = svd.fit_transform(tfidf_matrix)

tfidf_cols = [f"tfidf_{i}" for i in range(tfidf_reduced.shape[1])]
tfidf_df = pd.DataFrame(tfidf_reduced, columns=tfidf_cols, index=df.index)
df = pd.concat([df, tfidf_df], axis=1)

# Save vectorizer + svd for inference reproducibility
joblib.dump(tfidf, MODEL_DIR / "tfidf_vectorizer.joblib")
joblib.dump(svd, MODEL_DIR / "tfidf_svd.joblib")
print(f"✅ Saved TF-IDF & SVD artifacts to {MODEL_DIR}")

# ================================
# Finalize & save features
# ================================
base_cols = ["meeting_id", "speaker_id", "role"]
missing_base = [c for c in base_cols if c not in df.columns]
if missing_base:
    raise KeyError(f"Missing required columns: {missing_base}")

feature_cols = base_cols + [
    "word_count", "avg_sentence_len", "question_count",
    "hard_directive_count", "soft_help_count", "directive_count",
    "uncertainty_count", "is_greeting_only",
    "sentiment_score", "lm_manager_prob",
    "word_count_rel", "directive_count_rel", "word_count_share"
] + tfidf_cols

df_features = df[feature_cols].copy()
# coerce meta ids to string (avoid numeric surprises)
df_features["meeting_id"] = df_features["meeting_id"].astype(str)
df_features["speaker_id"] = df_features["speaker_id"].astype(str)

df_features.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Saved {len(df_features)} samples with {len(feature_cols)} features to {OUTPUT_PATH}")

feature_stats = df_features.select_dtypes(include=[np.number]).describe()
feature_stats.to_csv(FEATURE_STATS_CSV)
print(f"📊 Feature stats saved to {FEATURE_STATS_CSV}")

print("\n🔍 Feature preview:")
print(df_features.head(3))


✅ Groq client initialized
✅ Loaded 394 samples for feature extraction
🧠 Adding LLM feature (cached entries: 0)...
⚠️ Groq API error: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
⚠️ Groq API error: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
⚠️ Groq API error: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on w

KeyboardInterrupt: 